Create an evaluation pipeline that loads your LoRA-DAPT model and answers questions directly — no QLoRA, no FAISS.

🧠 Purpose:
This script evaluates the DAPT-only model using a LoRA adapter trained on full_university_data.txt. It does not include supervised fine-tuning (QLoRA) or RAG.

It represents Pipeline P2 in your project.

Step 1: Imports & Setup
Sets up model ID and adapter path.

Prepares imports to use Hugging Face Transformers and LoRA integration (peft).

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Paths
base_model_id = "meta-llama/Llama-2-7b-hf"
adapter_path = "../checkpoints/llama2_dapt_lora/"


Step 2: Load Tokenizer
Loads the tokenizer associated with LLaMA 2.

Sets pad_token to eos_token to avoid padding mismatch errors.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_auth_token=True)
tokenizer.pad_token = tokenizer.eos_token

c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\tokenization_auto.py:809: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Step 3: Load 4-bit Base Model + Attach LoRA-DAPT Adapter
Loads LLaMA 2 in 4-bit using BitsAndBytesConfig for memory savings.

Applies the LoRA-DAPT adapter from your unsupervised training (llama2_dapt_lora/).

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load 4-bit base model with offloading
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",                  # auto-dispatches across GPU/CPU
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_auth_token=True
)

# Attach LoRA DAPT adapter
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Li

Step 4: Define ask_dapt_model()
Formats a user question into an instruction-style prompt.

Uses generate() with sampling parameters (temperature, top_k, top_p) to produce a response.

In [5]:
def ask_dapt_model(question):
    prompt = f"""You are a helpful assistant at a university.

Answer the following student question clearly and helpfully.

Question: {question}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.split("Answer:")[-1].strip()


Step 5: Run Evaluation
Asks four example university questions.

Prints responses using only the DAPT-trained model.

In [6]:
questions = [
    "Where can I book a study room?",
    "What is the PGR Lounge?",
    "How do I contact IT services?",
    "Can I get help with academic writing?",
]

for q in questions:
    print(f"🧑‍🎓 Question: {q}")
    print("🤖 Answer:", ask_dapt_model(q))
    print("-" * 80)


🧑‍🎓 Question: Where can I book a study room?


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\llama\modeling_llama.py:602: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


🤖 Answer: You can book a study room on the University of Bradford website. You must have a University username and password to do this. You can also use a study room without booking one. This is called 'walk-in' availability. You can use a study room without booking one if it isn't booked by someone else. You can see which study rooms are available and which are booked on the study room website. You can also ask at the library reception desk. The library reception desk is on the ground floor. It's at the entrance of the library. You can also ask at the reception desk on the first floor. The reception desk on the first floor is on the landing of the stairs. There are two staircases in the library. There is one staircase on the ground floor. There is one staircase on the first floor. You can also ask at the reception desk on the second floor. The reception desk on the second floor is on the landing of the stairs. There is one staircase on the second floor. You can also ask at the recepti

✅ Suggestions for Improvement

Area	Suggestion
File name	Rename to evaluate_dapt_chatbot.ipynb
Output capture	Store results in a dictionary for later use in report
Prompt	Include a fixed prompt template to ensure fair comparison
Notebook section	Add a Markdown cell explaining: “This notebook evaluates Pipeline P2 (Unsupervised Only)”
✅ Pipeline Relevance
This script supports direct comparison against:


Pipeline	Description
P1	QLoRA only (supervised)
P2	✅ DAPT only (this script)
P3	DAPT + QLoRA merged
P4	Full pipeline with RAG